In [1]:
import tensorflow as tf
import sys

print(f"TensorFlow version: {tf.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')

if gpu_devices:
    print(f"SUCCESS: GPU Detected: {gpu_devices}")
else:
    print("WARNING: No GPU detected.")

TensorFlow version: 2.15.0
SUCCESS: GPU Detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# (Mount Drive, install libraries: tensorflow, etc.)
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import os

In [3]:
# --- 1. Load Preprocessed Segment Data ---
data = np.load('processed_data/emotify_spectrograms_5s_segments.npz')
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']

# --- 2. Define the Multi-Label CNN Model ---
input_shape = X_train.shape[1:]  # This will be (128, 216, 1) for 5-second segments
num_classes = y_train.shape[1] # Number of emotion columns

print(f"Input shape: {input_shape}")
print(f"Number of classes: {num_classes}")
print(f"Training segments: {X_train.shape[0]}")
print(f"Testing segments: {X_test.shape[0]}")
print(f"Label range: [{y_train.min():.3f}, {y_train.max():.3f}]")  # Should be 0-1 for weighted labels

Input shape: (128, 216, 1)
Number of classes: 9
Training segments: 7334
Testing segments: 1818
Label range: [0.000, 1.000]


In [4]:
X_train[1]

array([[[1.        ],
        [1.        ],
        [1.        ],
        ...,
        [1.        ],
        [1.        ],
        [1.        ]],

       [[1.        ],
        [1.        ],
        [1.        ],
        ...,
        [1.        ],
        [1.        ],
        [1.        ]],

       [[1.        ],
        [1.        ],
        [1.        ],
        ...,
        [1.        ],
        [1.        ],
        [1.        ]],

       ...,

       [[0.58615506],
        [0.59835804],
        [0.5844413 ],
        ...,
        [0.67059445],
        [0.6431416 ],
        [0.6054172 ]],

       [[0.547847  ],
        [0.5545715 ],
        [0.55434716],
        ...,
        [0.59696186],
        [0.5597463 ],
        [0.5814477 ]],

       [[0.4978289 ],
        [0.47386813],
        [0.4374916 ],
        ...,
        [0.39219743],
        [0.45202875],
        [0.56835264]]], dtype=float32)

In [5]:
y_train[1]

array([0.14285715, 0.07142857, 0.2857143 , 0.35714287, 0.5714286 ,
       0.        , 0.14285715, 0.21428572, 0.        ], dtype=float32)

In [6]:
from tensorflow.keras import layers, models, regularizers, Input
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

def create_crnn_model(input_shape, num_classes):
    inputs = Input(shape=input_shape) # Expected: (128, 216, 1)

    # --- CNN Feature Extractor ---
    # We use (2,2) pooling to reduce size, but switch to (2,1) later
    # to preserve the Time Dimension (Width).

    # Block 1
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    # Block 2
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)

    # Block 3
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)

    # Block 4
    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    # Pool Frequency (Height) only, keep Time (Width) intact here
    x = layers.MaxPooling2D((2, 1))(x)
    x = layers.Dropout(0.4)(x)

    # --- Reshape for RNN ---
    # Current shape is likely (Batch, Freq, Time, Filters)
    # We need (Batch, Time, Features) for the LSTM.

    # 1. Swap axes to put Time first: (Batch, Time, Freq, Filters)
    x = layers.Permute((2, 1, 3))(x)

    # 2. Reshape to flatten Freq and Filters into one "Feature" vector per time step
    # We interpret the shape dynamically to avoid hardcoding numbers
    s = x.shape
    x = layers.Reshape((-1, s[2] * s[3]))(x)

    # --- RNN Layers (The "Memory") ---
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(32))(x)

    # --- Dense Layers ---
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation='sigmoid')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="CRNN_Model")
    return model

# Create and Compile
model = create_crnn_model(input_shape, num_classes)

class MultiLabelF1(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', threshold=0.5, **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision_m = tf.keras.metrics.Precision(thresholds=threshold)
        self.recall_m = tf.keras.metrics.Recall(thresholds=threshold)

    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision_m.update_state(y_true, y_pred, sample_weight)
        self.recall_m.update_state(y_true, y_pred, sample_weight)

    def result(self):
        p = self.precision_m.result()
        r = self.recall_m.result()
        return 2 * ((p * r) / (p + r + tf.keras.backend.epsilon()))

    def reset_states(self):
        self.precision_m.reset_states()
        self.recall_m.reset_states()

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        MultiLabelF1(threshold=0.3), # Ensure your F1 metric class is defined
        tf.keras.metrics.AUC(name='auc', multi_label=True)
    ]
)

model.summary()

2025-12-15 17:35:12.667136: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2025-12-15 17:35:12.667159: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-12-15 17:35:12.667165: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-12-15 17:35:12.667218: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-12-15 17:35:12.667250: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "CRNN_Model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128, 216, 1)]     0         
                                                                 
 conv2d (Conv2D)             (None, 128, 216, 32)      320       
                                                                 
 batch_normalization (Batch  (None, 128, 216, 32)      128       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 64, 108, 32)       0         
 D)                                                              
                                                                 
 dropout (Dropout)           (None, 64, 108, 32)       0         
                                                                 
 conv2d_1 (Conv2D)           (None, 64, 108, 64)       1

In [7]:
import glob
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

checkpoint_dir = 'checkpoints/crnn_model/'
os.makedirs(checkpoint_dir, exist_ok=True)

# Improved checkpoint callback
checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, 'best_crnn_model.weights.h5'),
    monitor='val_f1_score',  # Monitor F1 score instead of loss
    save_best_only=True,
    save_weights_only=True,
    mode='max',  # We want to maximize F1 score
    verbose=1
)

# Additional callbacks for better training
early_stopping = EarlyStopping(
    monitor='val_f1_score',
    patience=20,  # More patience for segment-based training
    restore_best_weights=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_f1_score',
    factor=0.5,
    patience=8,
    min_lr=1e-7,
    mode='max',
    verbose=1
)

# Find latest checkpoint
checkpoints = glob.glob(os.path.join(checkpoint_dir, '*.weights.h5'))
if checkpoints:
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"Resuming from: {latest_checkpoint}")
    model.load_weights(latest_checkpoint)
    # Extract epoch from filename if possible, else start from 0
    try:
        initial_epoch = int(latest_checkpoint.split('epoch_')[1].split('.')[0])
    except:
        initial_epoch = 0
else:
    print("No checkpoint found. Training from scratch.")
    initial_epoch = 0

print(f"Starting from epoch: {initial_epoch}")

No checkpoint found. Training from scratch.
Starting from epoch: 0


In [8]:
import numpy as np
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import os

# --- 1. Calculate Class Weights (Crucial for Imbalanced Data) ---
def calculate_class_weights(y_train):
    """
    Calculate weights for each emotion class.
    Rare emotions get higher weights so the model doesn't ignore them.
    """
    # Sum the weighted probabilities for each class
    class_counts = np.sum(y_train, axis=0)
    total_samples = len(y_train)

    # Formula: total / (num_classes * class_count)
    # This balances the influence of each emotion.
    class_weights_array = total_samples / (len(class_counts) * class_counts)

    # Convert to dictionary {0: weight, 1: weight...} for Keras
    class_weights_dict = {i: weight for i, weight in enumerate(class_weights_array)}
    return class_weights_dict

# Calculate the weights
class_weights = calculate_class_weights(y_train)
print("Class weights:", class_weights)

# Create a specific directory for this new model
checkpoint_dir = 'checkpoints/crnn_model/'
os.makedirs(checkpoint_dir, exist_ok=True)
model_path = os.path.join(checkpoint_dir, 'best_crnn_model.keras') # .keras is the modern format

callbacks = [
    ModelCheckpoint(
        filepath=model_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)
]

print("Starting CRNN training...")
history = model.fit(
    X_train, y_train,
    epochs=60, # CRNNs sometimes take a little longer to converge
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    class_weight=class_weights, # Ensure class_weights is defined from previous steps
    verbose=1
)

Class weights: {0: 0.82999796, 1: 0.580072, 2: 0.59960455, 3: 0.4173088, 4: 0.3578358, 5: 0.6271168, 6: 0.42173347, 7: 0.4933882, 8: 0.5912981}
Starting CRNN training...
Epoch 1/60


2025-12-15 17:35:16.735045: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-12-15 17:35:16.919970: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


230/230 [==============================] - ETA: 0s - loss: 0.2634 - accuracy: 0.2786 - f1_score: 0.3622 - auc: 0.6107   

/Users/dhananjoyshaw/Desktop/MER/mer_env/lib/python3.11/site-packages/keras/src/engine/training.py:2723: UserWarning: Metric MultiLabelF1 implements a `reset_states()` method; rename it to `reset_state()` (without the final "s"). The name `reset_states()` has been deprecated to improve API consistency.
  m.reset_state()



Epoch 1: val_loss improved from inf to 0.52970, saving model to checkpoints/crnn_model/best_crnn_model.keras
230/230 [==============================] - 46s 156ms/step - loss: 0.2634 - accuracy: 0.2786 - f1_score: 0.3622 - auc: 0.6107 - val_loss: 0.5297 - val_accuracy: 0.1694 - val_f1_score: 0.4911 - val_auc: 0.5073 - lr: 0.0010
Epoch 2/60
230/230 [==============================] - ETA: 0s - loss: 0.2335 - accuracy: 0.3343 - f1_score: 0.3588 - auc: 0.6893 
Epoch 2: val_loss improved from 0.52970 to 0.51923, saving model to checkpoints/crnn_model/best_crnn_model.keras
230/230 [==============================] - 33s 145ms/step - loss: 0.2335 - accuracy: 0.3343 - f1_score: 0.3588 - auc: 0.6893 - val_loss: 0.5192 - val_accuracy: 0.1694 - val_f1_score: 0.3585 - val_auc: 0.5964 - lr: 0.0010
Epoch 3/60
230/230 [==============================] - ETA: 0s - loss: 0.2279 - accuracy: 0.3800 - f1_score: 0.3942 - auc: 0.7237 
Epoch 3: val_loss did not improve from 0.51923
230/230 [===================

In [9]:
# Predict on a single sample
# Add a batch dimension to the input data for prediction
input_sample = np.expand_dims(X_test[10], axis=0)
predictions = model.predict(input_sample)
print(predictions)

1/1 [==============================] - 2s 2s/step
[[0.1996432  0.27621543 0.15690923 0.21650726 0.31740236 0.17888753
  0.32146943 0.2537436  0.16487245]]


In [10]:
y_test[10]

array([0.27272728, 0.27272728, 0.27272728, 0.09090909, 0.36363637,
       0.45454547, 0.        , 0.27272728, 0.        ], dtype=float32)

In [11]:
# Evaluate the model on the test set
print("Evaluating the model on the test set...")
results = model.evaluate(X_test, y_test, batch_size=32, verbose=1)

# Print the evaluation results
print("Test Loss:", results[0])
# Assuming the order of metrics in model.compile matches the results list
metric_names = model.metrics_names
for name, value in zip(metric_names, results):
    print(f"Test {name}: {value}")

Evaluating the model on the test set...
57/57 [==============================] - 4s 63ms/step - loss: 0.4900 - accuracy: 0.3751 - f1_score: 0.4660 - auc: 0.7831
Test Loss: 0.49003779888153076
Test loss: 0.49003779888153076
Test accuracy: 0.3751375079154968
Test f1_score: 0.46601396799087524
Test auc: 0.7831246256828308


In [12]:
results

[0.49003779888153076,
 0.3751375079154968,
 0.46601396799087524,
 0.7831246256828308]

In [19]:
import os

# Define the path to save the model within your Google Drive project directory
save_dir = 'saved_model'
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, 'final_crnn_model.keras')

# Save the final CRNN model
model.save(model_path)
print(f"Model saved to: {model_path}")

Model saved to: saved_model/final_crnn_model.keras
